# This file is a quick reference demo on how to utilize the fuzzycar library for the rc car 

# # Getting Started With Overlay

Whenever you are creating a new python file , always make sure you import the pynq library. We will also need Overlay and MMIO as fuzzycar depends on them.
From the fuzzycar library, we will need UartAXI, SPIController, and PWMController

In [1]:
from pynq import Overlay , MMIO
from fuzzycar import UartAXI , SPIController , PWMController

Once you have imported these libraries, the next step is to actually put the bitstream onto the FPGA. The bitstream overlay will then become an object we can utilize in our project. Standard convention is to name the Overlay variable ol. 

In [2]:
ol = Overlay('car.bit')

The Overlay class from pynq handles all of the organzing of the bitstream. When you call on it, it will search for an .hwh file that tells it everything inside of the bitstream. From there, you can utilize new functions to utilize the ip blocks in our project.

For example, Overlay makes a dictionary of all available ip blocks and their respective relevant data, you can find this by calling on:

Upon running that function, you can see there is a LOT of data Overlay can pull. Overlay  allows us to automatically find the correct addressing for all the IP blocks. We can then create each object of the car. Instead of sifting through the dictionary, you can utilize python syntax to call on the IP block if you already know it's name created in Vivado. For our project, our bitstream contains the following IP blocks:

Passenger : Uart
Driver : Uart
Front : Uart
Lora : SPI
Motor : AXI Timer (PWM)

In order to comminucate with each IP block, we need to figure out its address. Overlay allows us to automatically find its address. For example, if we call on:

In [3]:
hex(ol.passenger_side.mmio.base_addr)

'0x42c20000'

You can see that Overlay has found the address that starts the ip block for the passenger uart IP block ( sonar sensor). Now that we are able to easily find the correct addresses for our project, we need to now configure and talk to the IP blocks.

# Using the fuzzycar library

 The FuzzCar library handles all the register values and required set up. Using the fuzzycar library is as such:

First, you need to assign a variable (object) to become that specific instance of a device controller. For quick reference, copy this into your work to ensure variable names are consisent. 

In [6]:
passenger = UartAXI(ol.passenger_side.mmio.base_addr)
driver = UartAXI(ol.driver_side.mmio.base_addr)
front = UartAXI(ol.front_side.mmio.base_addr)
lora = SPIController(ol.lora.mmio.base_addr)
steering = PWMController(ol, 'axi_timer_0')
motor = PWMController(ol, 'pwm')

Using the UartAXI class is the same way, let's say you wanted to figure out what the driver side door sensor is currently reporting. You can call on

In [11]:
steering.set_pwm(50,0)

Uh Oh, it tossed an error, for Uart interfaces you must always tell it how many bytes you want to read. 

In [6]:
passenger.read(1)

[]

# Using The Sensor Classes

Now that you have created controllers for each component, the next step is to utilize the sensor functions to read and parse the data. This is how you can read the outputs of the Sonar sensors.

First, import the maxsonar class from fuzzycar

In [4]:
from fuzzycar import MAXSONAR 

The maxsonar class is designed to handle the uart data that comes out of the UartAXI controller. It automatically converts the recieved bits into inches.   

Let's say we want to read the driver side's sensor data. Create the maxsonar object by following this syntax

In [7]:
driver_sonar = MAXSONAR(driver)

If we want to view the maxsonar class, we can use the ?? syntax 

In [28]:
driver_sonar.read_distance()

43

The sonar modules for this project auto calibrate and will continuously spew data into the FIFO, it is set to automatically overwrite itself when it gets full. Be careful, and make sure you are constantly reading from this FIFO!

Let's try to read data from the sonar fifo.

In [104]:
import time
from IPython.display import clear_output

start_time = time.time()

while (time.time() - start_time) < 40:  # Run for 40 seconds
    response = lora.transfer([0x00])  # Send request to LoRa
    
    # Convert list of integers to bytes and decode
    received_str = ''.join(chr(b) for b in response if 32 <= b <= 126)  # Keep printable ASCII

    clear_output(wait=True)  # Clear Jupyter console

    if 'I' in received_str:
        print("Received: I")
    elif 'L' in received_str:
        print("Received: L")
    elif 'R' in received_str:
        print("Received: R")

    time.sleep(0.5)  # Small delay to prevent overwhelming the console


Received: L


The read_distance() function takes no arguments, and returns an integer that represents the closest object to the sensor in inches. You can now take this value and process it as needed. 

Let's now look at an example in python how we can have these components work together. This code will show a simple if then decison tree on if we should accelerate or not. It will also show you some of the Juypter IPython display abilities.

In [66]:
from IPython.display import clear_output
import time

start_time = time.time()  # Record the start time

while time.time() - start_time < 10:  # Run the loop for 10 seconds
    
    clear_output(wait=True)  # Clear the console output
    distance = driver_sonar.read_distance()
    
    if distance > 10 :  # If the car is more than 40 inches away from anything
        steering.set_pwm_time(50,1680)
        lora.read(1)
        print(f"Distance: {distance} inches - Motor running")
    else:
        lora.
        steering.set_pwm_time(50,1600)
        print(f"Distance: {distance} inches - Motor stopped")
    
    time.sleep(0.1)  # Add a short delay to prevent rapid clearing



PWM configured: Frequency = 50 Hz, Duty Cycle = 1%
Distance: 84 inches - Motor running


In [ ]:
lora.transfer([0x00])

PWM configured: Frequency = 50 Hz, Duty Cycle = 10%
